# 02. Профилирование и очистка данных

**ЛР 2, шаг 7.** Цель: понять, насколько данные пригодны для ответа на вопросы заказчика, найти проблемы и подготовить очищенные слои в `data/processed/`.

Что проверяем по каждому слою:
- число объектов и типы геометрии;
- систему координат (CRS) и охват;
- пустые и некорректные геометрии;
- дубликаты (по идентификатору и по положению);
- заполненность атрибутов;
- объекты за пределами территории.

Найденные проблемы оформляйте задачами с меткой `data` и переносите выводы в `docs/data-quality.md`.\n\n[![Открыть в Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OWNER/REPO/blob/main/notebooks/02_data_profiling.ipynb)\n\nЧтобы кнопка открывала ваш ноутбук, замените в ссылке `OWNER/REPO` на владельца и имя своего репозитория (например, `ivan-p/pm-lab-00-aquamap`). Другие способы открыть ноутбук описаны в приложении Б.

## Настройка среды

Ноутбук рассчитан на **Google Colab** (как получить бесплатный доступ: приложение Б к заданию ЛР 2). Он работает и локально, в Jupyter или VS Code.

Что делает ячейка ниже:
1. В Colab ставит недостающие библиотеки и подключает Google Диск. Colab попросит разрешение на доступ к Диску: выберите тот же аккаунт Google, под которым открыт Colab.
2. Находит рабочую папку группы на Диске. Все данные и результаты сохраняются туда, поэтому после отключения сессии ничего не пропадёт.
3. Создаёт внутри неё ту же структуру папок, что и в репозитории: `data/raw`, `data/manual`, `data/processed`, `docs/img`, `docs/maps`.

**Перед первым запуском** впишите в `PROJECT_FOLDER` имя папки группы на Google Диске.

Запуск всего ноутбука: **Среда выполнения → Выполнить все** (Runtime → Run all).

In [ ]:
# === Настройка среды: запустите эту ячейку первой ===
import sys, subprocess
from pathlib import Path

# Имя папки группы в «Моём диске» Google. Обычно совпадает с именем репозитория.
PROJECT_FOLDER = "pm-lab-00-aquamap"

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", 'geopandas>=1.0'], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive") / PROJECT_FOLDER
    if not BASE.exists():
        raise FileNotFoundError(
            f"На Google Диске нет папки «{PROJECT_FOLDER}». Создайте её или добавьте ярлык "
            "общей папки группы в «Мой диск» (приложение Б, шаг 2) и запустите ячейку снова.")
else:
    BASE = Path("..")   # локальный запуск из папки notebooks/ репозитория

RAW, MANUAL, PROCESSED = BASE / "data" / "raw", BASE / "data" / "manual", BASE / "data" / "processed"
IMG, MAPS = BASE / "docs" / "img", BASE / "docs" / "maps"
for folder in (RAW, MANUAL, PROCESSED, IMG, MAPS):
    folder.mkdir(parents=True, exist_ok=True)
print("Среда:", "Google Colab" if IN_COLAB else "локальная", "· рабочая папка:", BASE.resolve())

In [ ]:
import geopandas as gpd
import pandas as pd

LAYERS = ["districts", "water_points", "parks", "residential"]   # слои из ноутбука 01
DUPLICATE_DISTANCE_M = 5          # точки ближе этого расстояния считаем возможными дублями

boundary = gpd.read_file(RAW / "boundary.gpkg")
layers = {name: gpd.read_file(RAW / f"{name}.gpkg") for name in LAYERS}
METRIC_CRS = boundary.estimate_utm_crs()   # метрическая проекция UTM для расстояний и площадей
print("Метрическая система координат:", METRIC_CRS)

## Сводный профиль слоёв

In [ ]:
def profile(name: str, gdf: gpd.GeoDataFrame) -> dict:
    g = gdf.geometry
    inside = gdf.intersects(boundary.to_crs(gdf.crs).union_all())
    dup_id = gdf.duplicated(subset=["osm_type", "osm_id"]).sum() if "osm_id" in gdf else None
    minx, miny, maxx, maxy = gdf.total_bounds
    return {
        "слой": name,
        "объектов": len(gdf),
        "CRS": gdf.crs.to_string() if gdf.crs else "НЕ ЗАДАНА",
        "типы геометрии": ", ".join(f"{k}: {v}" for k, v in g.geom_type.value_counts().items()),
        "пустых геометрий": int(g.is_empty.sum() + g.isna().sum()),
        "некорректных геометрий": int((~g.is_valid).sum()),
        "дубликатов по id": dup_id,
        "вне территории": int((~inside).sum()),
        "охват (lon/lat)": f"{minx:.3f}, {miny:.3f} – {maxx:.3f}, {maxy:.3f}",
    }

summary = pd.DataFrame([profile(n, g) for n, g in layers.items()])
summary.to_csv(PROCESSED / "profile_summary.csv", index=False)
summary

## Заполненность атрибутов

Доля непустых значений по каждому атрибуту. Низкая заполненность важного атрибута (например, названия или режима работы) — повод для задачи или для пересмотра требований.

In [ ]:
completeness = {}
for name, gdf in layers.items():
    attrs = gdf.drop(columns=["geometry", "osm_type", "osm_id"], errors="ignore")
    completeness[name] = (attrs.notna().mean() * 100).round(1)
completeness = pd.DataFrame(completeness).fillna("—")
completeness

## Точки: дубликаты по положению и подозрительные объекты

Для точечного слоя ищем объекты, стоящие ближе `DUPLICATE_DISTANCE_M` метров друг от друга, и смотрим на сочетания тегов. В примере AquaMap часть объектов `amenity=drinking_water` одновременно отмечена как `man_made=water_well` (колодец или скважина): это другой тип объекта, и заказчику нужно решить, учитывать ли их.

In [ ]:
pts = layers["water_points"].copy()
pts["geometry"] = pts.geometry.representative_point()      # полигоны и линии превращаем в точки
pts_m = pts.to_crs(METRIC_CRS)

near = gpd.sjoin_nearest(pts_m, pts_m[["geometry"]], how="left",
                         max_distance=DUPLICATE_DISTANCE_M, distance_col="dist", exclusive=True)
near_dups = near[near["dist"].notna()]
print(f"Точек с соседом ближе {DUPLICATE_DISTANCE_M} м: {near_dups.index.nunique()}")

tag_combos = pts.groupby(["amenity", "man_made"], dropna=False).size().rename("объектов")
tag_combos

## Очистка

Правила очистки — это решения, которые влияют на результат анализа. Запишите каждое правило в `docs/data-quality.md` и согласуйте с заказчиком спорные случаи.

In [ ]:
EXCLUDE_MAN_MADE = ["water_well"]     # пример правила: колодцы и скважины не считаем питьевыми точками

clean = {}
area = boundary.union_all()
for name, gdf in layers.items():
    g = gdf[~gdf.geometry.is_empty & gdf.geometry.notna()].copy()
    g["geometry"] = g.geometry.make_valid()
    if "osm_id" in g:
        g = g.drop_duplicates(subset=["osm_type", "osm_id"])
    if name != "districts":
        g = g[g.intersects(area)]
    clean[name] = g

wp = clean["water_points"]
wp["geometry"] = wp.geometry.representative_point()
if "man_made" in wp:
    wp = wp[~wp["man_made"].isin(EXCLUDE_MAN_MADE)]
clean["water_points"] = wp

clean["boundary"] = boundary
for name, g in clean.items():
    g.to_file(PROCESSED / f"{name}.gpkg", driver="GPKG")

pd.DataFrame({"было": {n: len(layers[n]) for n in LAYERS},
              "стало": {n: len(clean[n]) for n in LAYERS}})

## Карта для визуальной проверки

In [ ]:
ax = clean["districts"].plot(figsize=(9, 9), color="whitesmoke", edgecolor="gray")
clean["parks"].plot(ax=ax, color="lightgreen")
clean["water_points"].plot(ax=ax, color="blue", markersize=6)
ax.set_title("Очищенные данные (© участники OpenStreetMap)")
ax.set_axis_off()

## Перенос результатов в репозиторий

Очищенные слои из `data/processed` должны попасть в репозиторий через Pull Request. В Colab ячейка ниже соберёт их в архив `results_02.zip` и скачает на компьютер. Распакуйте архив и загрузите файлы в те же папки репозитория в своей ветке: **Add file → Upload files** (подробно в приложении Б, шаг 6).

Файлы больше 10 МБ в репозиторий не загружайте: ячейка о них предупредит. При локальной работе результаты уже лежат в папках репозитория, ячейку можно пропустить.

In [ ]:
import shutil
FOLDERS = [PROCESSED]
if IN_COLAB:
    from google.colab import files
    pack = Path("/content/results_02")
    shutil.rmtree(pack, ignore_errors=True)
    for folder in FOLDERS:
        for f in folder.rglob("*"):
            if f.is_file() and f.name != ".gitkeep":
                if f.stat().st_size > 10 * 1024 * 1024:
                    print(f"Пропущен (больше 10 МБ): {f.relative_to(BASE)}")
                    continue
                target = pack / f.relative_to(BASE)
                target.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(f, target)
    archive = shutil.make_archive(str(pack), "zip", pack)
    print("В архиве:", sorted(str(p.relative_to(pack)) for p in pack.rglob("*") if p.is_file()))
    files.download(archive)
else:
    print("Локальный запуск: результаты уже лежат в папках репозитория")

## Выводы (заполните)

- Пригодны ли данные для ответа на каждый аналитический вопрос?
- Какие проблемы найдены и какие задачи заведены (номера Issues)?
- Какие правила очистки применены и согласованы ли они с заказчиком?

Перенесите выводы в `docs/data-quality.md`.